In [ ]:
!pip install torch torchvision matplotlib tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import os
import zipfile
from tqdm import tqdm


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/Colab Notebooks/archive (13).zip'
extract_path = '/content/food101'

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
    print("Распаковано в", extract_path)


Распаковано в /content/food101


In [ ]:
!ls /content/food101


food-101


In [ ]:
!ls /content/food101/food-101/


images	license_agreement.txt  meta  README.txt


In [ ]:
import os
classes = os.listdir('/content/food101/food-101/images')
print(f"Количество классов: {len(classes)}")
print("Первые 10 классов:", classes[:10])

Количество классов: 101
Первые 10 классов: ['creme_brulee', 'falafel', 'carrot_cake', 'scallops', 'poutine', 'strawberry_shortcake', 'takoyaki', 'samosa', 'prime_rib', 'bruschetta']


In [ ]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# Правильный путь к папке с изображениями
data_dir = '/content/food101/food-101/images'


In [ ]:
# Трансформации для тренировки (с аугментацией)
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Трансформации для валидации (без аугментации)
transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [ ]:
# Загружаем весь датасет
full_dataset = datasets.ImageFolder(data_dir, transform=transform_train)

print(f"Всего изображений: {len(full_dataset)}")
print(f"Количество классов: {len(full_dataset.classes)}")

Всего изображений: 101000
Количество классов: 101


In [ ]:
dataset_size = len(full_dataset)
train_size = int(0.8 * dataset_size)
val_size = dataset_size - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Для валидации убираем аугментацию (важно!)
val_dataset.dataset.transform = transform_val

batch_size = 64  # можно уменьшить до 32, если не хватает памяти GPU
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print(f"Тренировочных батчей: {len(train_loader)}")
print(f"Валидационных батчей: {len(val_loader)}")

Тренировочных батчей: 1263
Валидационных батчей: 316


In [ ]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Устройство: {device}")

# Загружаем предобученную EfficientNet-B0
model = models.efficientnet_b0(weights='IMAGENET1K_V1')

# Заменяем последний слой на 101 класс
num_classes = 101
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

num_epochs = 10
best_acc = 0.0

for epoch in range(num_epochs):
    # Тренировка
    model.train()
    running_loss = 0.0
    train_correct = 0
    train_total = 0

    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Train]')
    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

        loop.set_postfix(loss=loss.item(), acc=train_correct/train_total)

    train_acc = train_correct / train_total

    # Валидация
    model.eval()
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f'Epoch {epoch+1}/{num_epochs} [Val]'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = val_correct / val_total
    print(f'Epoch {epoch+1}: Loss: {running_loss/len(train_loader):.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}')


    # Сохраняем лучшую модель на Google Диск
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), '/content/drive/MyDrive/best_food_classifier.pth')
        print(f'Сохранена лучшая модель с точностью {best_acc:.4f}')

    scheduler.step()

print(f'Обучение завершено. Лучшая точность на валидации: {best_acc:.4f}')

Устройство: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 179MB/s]
Epoch 1/10 [Val]: 100%|██████████| 316/316 [01:43<00:00,  3.06it/s]


Epoch 1: Loss: 1.5524, Train Acc: 0.6040, Val Acc: 0.7062
Сохранена лучшая модель с точностью 0.7062


Epoch 2/10 [Val]: 100%|██████████| 316/316 [01:33<00:00,  3.39it/s]


Epoch 2: Loss: 0.9505, Train Acc: 0.7422, Val Acc: 0.7380
Сохранена лучшая модель с точностью 0.7380


...

Epoch 8: Loss: 0.0970, Train Acc: 0.9735, Val Acc: 0.8070


Epoch 9/10 [Val]: 100%|██████████| 316/316 [01:34<00:00,  3.36it/s]


Epoch 9: Loss: 0.0694, Train Acc: 0.9819, Val Acc: 0.8083
Сохранена лучшая модель с точностью 0.8083


Epoch 10/10 [Val]: 100%|██████████| 316/316 [01:32<00:00,  3.41it/s]

Epoch 10: Loss: 0.0529, Train Acc: 0.9865, Val Acc: 0.8043
Обучение завершено. Лучшая точность на валидации: 0.8083


In [ ]:
!pip install onnx onnxruntime onnx2tf tensorflow


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.2/223.2 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 73.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.3/16.3 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 95.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 689.1/689.1 kB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB

In [ ]:
!pip install onnx onnxruntime onnx2tf tensorflow --quiet

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# Загружаем модель PyTorch
model = models.efficientnet_b0(weights=None)
num_classes = 101
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model_path = '/content/drive/MyDrive/best_food_classifier.pth'
model.load_state_dict(torch.load(model_path, map_location='cpu'))
model.eval()

# Экспорт в ONNX
dummy_input = torch.randn(1, 3, 224, 224)
onnx_path = '/content/drive/MyDrive/food_classifier.onnx'
torch.onnx.export(model, dummy_input, onnx_path,
                  input_names=['input'], output_names=['output'],
                  dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}})
print("ONNX сохранён")

# Конвертация ONNX → TFLite (с помощью onnx2tf)
from onnx2tf import convert
convert(
    input_onnx_file_path=onnx_path,
    output_folder_path='/content/drive/MyDrive/tflite_model',
    output_integer_quantized_tflite=True,
    quant_type="per_tensor",
    allow_custom_ops=True
)
print("TFLite модель сохранена")

/tmp/ipykernel_2994/4121829156.py:18: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(model, dummy_input, onnx_path,
W0417 17:34:12.575000 2994 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0417 17:34:12.577000 2994 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0417 17:34:12.579000 2994 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int

[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
Applied 99 of general pattern rewrite rules.
ONNX сохранён


TypeError: convert() got an unexpected keyword argument 'allow_custom_ops'

In [ ]:
from onnx2tf import convert

convert(
    input_onnx_file_path=onnx_path,
    output_folder_path='/content/drive/MyDrive/tflite_model',
    output_integer_quantized_tflite=True,   # квантование int8
    quant_type="per_tensor",                # тип квантования
)
print("TFLite модель сохранена")


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add        │ 9              │ 9                │
│ Concat     │ 1              │ 0                │
│ Constant   │ 166            │ 166              │
│ Conv       │ 81             │ 81               │
│ Gemm       │ 1              │ 1                │
│ Mul        │ 65             │ 65               │
│ ReduceMean │ 17             │ 17               │
│ Reshape    │ 1              │ 1                │
│ Shape      │ 1              │ 0                │
│ Sigmoid    │ 65             │ 65               │
│ Model Size │ 16.3MiB        │ 15.7MiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ 

flatbuffer_direct lowering:   0%|          | 0/239 [00:00<?, ?it/s]

flatbuffer_direct post-lowering:   0%|          | 0/6 [00:00<?, ?it/s]

flatbuffer_direct export:   0%|          | 0/7 [00:00<?, ?it/s]

flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.129s serialize=0.085s (sanitize=0.001s build=0.015s pack=0.066s output=0.003s) write=0.042s size=15.75MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.086s serialize=0.063s (sanitize=0.001s build=0.009s pack=0.053s output=0.001s) write=0.022s size=7.90MB


flatbuffer_direct lowering:   0%|          | 0/239 [00:00<?, ?it/s]

flatbuffer_direct post-lowering:   0%|          | 0/6 [00:00<?, ?it/s]

flatbuffer_direct export:   0%|          | 0/7 [00:00<?, ?it/s]

flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.152s serialize=0.107s (sanitize=0.001s build=0.024s pack=0.078s output=0.004s) write=0.044s size=15.75MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.100s serialize=0.072s (sanitize=0.001s build=0.010s pack=0.060s output=0.001s) write=0.027s size=7.90MB


ValueError: flatbuffer_direct quant_type must be one of ["per-channel", "per-tensor"]. got: per_tensor

In [ ]:
from onnx2tf import convert

convert(
    input_onnx_file_path=onnx_path,
    output_folder_path='/content/drive/MyDrive/tflite_model',
    output_integer_quantized_tflite=True,
    quant_type="per-tensor",   # <- дефис!
)
print("TFLite модель сохранена")


Model optimizing started ============================================================
Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Add        │ 9              │ 9                │
│ Constant   │ 166            │ 166              │
│ Conv       │ 81             │ 81               │
│ Gemm       │ 1              │ 1                │
│ Mul        │ 65             │ 65               │
│ ReduceMean │ 17             │ 17               │
│ Reshape    │ 1              │ 1                │
│ Sigmoid    │ 65             │ 65               │
│ Model Size │ 15.7MiB        │ 15.7MiB          │
└────────────┴────────────────┴──────────────────┘

Simplifying...
Finish! Here is the difference:
┏━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃            ┃ Original Model ┃ Simplified Model ┃
┡━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ 

flatbuffer_direct lowering:   0%|          | 0/239 [00:00<?, ?it/s]

flatbuffer_direct post-lowering:   0%|          | 0/6 [00:00<?, ?it/s]

flatbuffer_direct export:   0%|          | 0/7 [00:00<?, ?it/s]

flatbuffer_direct write timing: stage=float32 mode=builder_direct total=0.138s serialize=0.093s (sanitize=0.001s build=0.019s pack=0.071s output=0.003s) write=0.043s size=15.75MB
flatbuffer_direct write timing: stage=float16 mode=builder_direct total=0.093s serialize=0.066s (sanitize=0.001s build=0.010s pack=0.055s output=0.001s) write=0.027s size=7.90MB
flatbuffer_direct write timing: stage=integer_quant mode=builder_direct total=0.114s serialize=0.060s (sanitize=0.001s build=0.005s pack=0.051s output=0.003s) write=0.053s size=4.07MB
flatbuffer_direct write timing: stage=full_integer_quant mode=builder_direct total=0.070s serialize=0.054s (sanitize=0.001s build=0.005s pack=0.048s output=0.000s) write=0.015s size=4.07MB
flatbuffer_direct write timing: stage=integer_quant_with_int16_act mode=builder_direct total=0.503s serialize=0.469s (sanitize=0.001s build=0.005s pack=0.462s output=0.001s) write=0.033s size=4.07MB
flatbuffer_direct write timing: stage=full_integer_quant_with_int16_act

In [ ]:
!wget https://raw.githubusercontent.com/roadkill01/food-101/master/meta/classes.txt -O /content/drive/MyDrive/labels.txt

--2026-04-17 17:40:13--  https://raw.githubusercontent.com/roadkill01/food-101/master/meta/classes.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-17 17:40:13 ERROR 404: Not Found.



In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/tflite_model/food_classifier_integer_quant.tflite')
files.download('/content/drive/MyDrive/labels.txt')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!wget https://raw.githubusercontent.com/roadkill01/food-101/master/meta/classes.txt -O /content/labels.txt
from google.colab import files
files.download('/content/labels.txt')

--2026-04-23 11:21:46--  https://raw.githubusercontent.com/roadkill01/food-101/master/meta/classes.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-04-23 11:21:46 ERROR 404: Not Found.



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>